In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Qiskit AI 기반 트랜스파일러 서비스 소개

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*예상 QPU 사용량: 없음 (참고: 이 튜토리얼은 트랜스파일에 초점을 맞추므로 작업을 실행하지 않습니다)*

## 배경
**Qiskit AI 기반 트랜스파일러 서비스(QTS)**는 라우팅 및 합성 패스 모두에 머신 러닝 기반 최적화를 도입합니다. 이 AI 모드는 특히 대규모 Circuit과 복잡한 하드웨어 토폴로지에서 기존 트랜스파일의 한계를 극복하도록 설계되었습니다.

**2025년 7월** 기준으로, **트랜스파일러 서비스**는 새로운 IBM Quantum&reg; 플랫폼으로 마이그레이션되었으며 더 이상 이전 방식으로 사용할 수 없습니다. 트랜스파일러 서비스 상태에 관한 최신 업데이트는 [트랜스파일러 서비스 문서](/guides/qiskit-transpiler-service)를 참고하세요. AI 트랜스파일러는 표준 Qiskit 트랜스파일과 유사하게 로컬에서 계속 사용할 수 있습니다. `generate_preset_pass_manager()`를 `generate_ai_pass_manager()`로 교체하기만 하면 됩니다. 이 함수는 AI 기반 라우팅 및 합성 패스를 로컬 트랜스파일 워크플로우에 직접 통합하는 패스 매니저를 구성합니다.

### AI 패스의 주요 기능
- 라우팅 패스: AI 기반 라우팅은 특정 Circuit과 Backend에 따라 qubit 경로를 동적으로 조정하여 과도한 SWAP Gate의 필요성을 줄입니다.
    - `AIRouting`: 레이아웃 선택 및 Circuit 라우팅

- 합성 패스: AI 기술은 다중 qubit Gate의 분해를 최적화하여 오류가 발생하기 쉬운 두 qubit Gate의 수를 최소화합니다.
    - `AICliffordSynthesis`: Clifford Gate 합성
    - `AILinearFunctionSynthesis`: 선형 함수 Circuit 합성
    - `AIPermutationSynthesis`: 순열 Circuit 합성
    - `AIPauliNetworkSynthesis`: Pauli Network Circuit 합성 (Qiskit Transpiler Service에서만 사용 가능하며 로컬 환경에서는 지원되지 않음)

- 기존 트랜스파일과의 비교: 표준 Qiskit Transpiler는 다양한 양자 Circuit을 효과적으로 처리할 수 있는 강력한 도구입니다. 그러나 Circuit의 규모가 커지거나 하드웨어 구성이 복잡해지면 AI 패스가 추가적인 최적화 이점을 제공할 수 있습니다. 라우팅 및 합성에 학습된 모델을 활용함으로써 QTS는 Circuit 레이아웃을 더욱 정교하게 조정하고 까다롭거나 대규모 양자 작업의 오버헤드를 줄입니다.

이 튜토리얼은 라우팅 패스와 합성 패스 모두를 사용하여 AI 모드를 평가하고, 기존 트랜스파일과의 결과를 비교하여 AI가 성능 향상을 제공하는 부분을 강조합니다.

사용 가능한 AI 패스에 대한 자세한 내용은 [AI 패스 문서](/guides/ai-transpiler-passes)를 참고하세요.

### 양자 Circuit 트랜스파일에 AI를 사용하는 이유
양자 Circuit이 크기와 복잡성 면에서 성장함에 따라, 기존 트랜스파일 방법은 레이아웃을 최적화하고 Gate 수를 효율적으로 줄이는 데 어려움을 겪습니다. 특히 수백 개의 Qubit을 포함하는 대규모 Circuit은 장치 제약, 제한된 연결성 및 qubit 오류율로 인해 라우팅 및 합성에 상당한 어려움을 부과합니다.

바로 이 부분에서 AI 기반 트랜스파일이 잠재적인 해결책을 제공합니다. 머신 러닝 기술을 활용하여 Qiskit의 AI 기반 Transpiler는 qubit 라우팅 및 Gate 합성에 대해 더 스마트한 결정을 내릴 수 있어 대규모 양자 Circuit의 최적화가 향상됩니다.

### 간략한 벤치마킹 결과
![Qiskit 대비 AI Transpiler 성능을 보여주는 그래프](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

벤치마킹 테스트에서 AI Transpiler는 표준 Qiskit Transpiler에 비해 일관되게 더 얕고 품질이 높은 Circuit을 생성했습니다. 이 테스트에서는 [`generate_preset_passmanager`]로 구성된 Qiskit의 기본 패스 매니저 전략을 사용했습니다. 이 기본 전략은 종종 효과적이지만 더 크거나 복잡한 Circuit에서는 어려움을 겪을 수 있습니다. 반면에 AI 기반 패스는 IBM Quantum 하드웨어의 heavy-hex 토폴로지로 트랜스파일할 때 대규모 Circuit(100개 이상의 qubit)에서 평균 24%의 두 qubit Gate 수 감소와 36%의 Circuit 깊이 감소를 달성했습니다. 이 벤치마크에 대한 자세한 내용은 이 [블로그](https://www.ibm.com/quantum/blog/qiskit-performance)를 참고하세요.

이 튜토리얼은 AI 패스의 주요 이점과 기존 방법과의 비교를 살펴봅니다.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## 요구 사항

이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:

* Qiskit SDK v1.0 이상, [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 이상
* AI 로컬 모드가 포함된 Qiskit IBM&reg; Transpiler(`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## 설정

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts: {qubit_range_sim}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Part I. Qiskit 패턴
이제 Qiskit 패턴을 사용하여 간단한 양자 Circuit으로 AI Transpiler 서비스를 사용하는 방법을 살펴보겠습니다. 핵심은 표준 `generate_preset_pass_manager()` 대신 `generate_ai_pass_manager()`로 `PassManager`를 생성하는 것입니다.
## Step 1: 고전적 입력을 양자 문제로 매핑
이 섹션에서는 `efficient_su2` Circuit, 즉 널리 사용되는 하드웨어 효율적 ansatz에서 AI Transpiler를 테스트합니다. 이 Circuit은 변분 양자 알고리즘(예: VQE) 및 양자 머신 러닝 작업에 특히 관련이 있어 트랜스파일 성능 평가를 위한 이상적인 테스트 케이스입니다.

`efficient_su2` Circuit은 단일 qubit 회전과 CNOT 같은 엔탱글링 Gate의 교대 레이어로 구성됩니다. 이 레이어는 Gate 깊이를 관리 가능한 수준으로 유지하면서 양자 상태 공간을 유연하게 탐색할 수 있게 합니다. 이 Circuit을 최적화함으로써 Gate 수를 줄이고, 충실도를 높이며, 노이즈를 최소화하는 것을 목표로 합니다. 이는 AI Transpiler의 효율성을 테스트하기에 좋은 후보가 됩니다.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### AI 및 기존 패스 매니저 생성
AI Transpiler의 효과를 평가하기 위해 두 번의 트랜스파일 실행을 수행합니다. 먼저 AI Transpiler를 사용하여 Circuit을 트랜스파일합니다. 그런 다음 AI Transpiler 없이 기존 방법을 사용하여 동일한 Circuit을 트랜스파일하여 비교를 실행합니다. 두 트랜스파일 프로세스 모두 선택한 Backend의 동일한 커플링 맵을 사용하고, 공정한 비교를 위해 최적화 레벨을 3으로 설정합니다.

이 두 방법 모두 Qiskit에서 Circuit을 트랜스파일하기 위한 `PassManager` 인스턴스를 생성하는 표준 접근 방식을 반영합니다.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

이 테스트에서는 efficient_su2 Circuit에서 AI Transpiler와 표준 트랜스파일 방법의 성능을 비교합니다. AI Transpiler는 비슷한 Gate 수를 유지하면서 눈에 띄게 더 얕은 Circuit 깊이를 달성합니다.

- **Circuit 깊이:** AI Transpiler는 더 낮은 두 qubit 깊이를 가진 Circuit을 생성합니다. AI 패스는 qubit 상호작용 패턴을 학습하고 규칙 기반 휴리스틱보다 하드웨어 연결성을 더 효과적으로 활용하여 깊이를 최적화하도록 훈련되어 있으므로 이는 예상된 결과입니다.

- **Gate 수:** 총 Gate 수는 두 방법 사이에서 비슷하게 유지됩니다. 표준 SABRE 기반 트랜스파일은 Gate 오버헤드를 지배하는 SWAP 수를 명시적으로 최소화하기 때문에 이는 예상과 일치합니다. AI Transpiler는 대신 전체 깊이를 우선시하며 더 짧은 실행 경로를 위해 때로 몇 가지 추가 Gate를 교환할 수 있습니다.

- **트랜스파일 시간:** AI Transpiler는 표준 방법보다 실행 시간이 더 오래 걸립니다. 이는 라우팅 및 합성 중에 학습된 모델을 호출하는 추가적인 계산 비용 때문입니다. 반면 SABRE 기반 Transpiler는 Rust로 재작성되고 최적화된 후 크게 빨라져 대규모에서 매우 효율적인 휴리스틱 라우팅을 제공합니다.

이러한 결과는 하나의 Circuit에 기반한 것임을 명심하는 것이 중요합니다. AI Transpiler와 기존 방법을 포괄적으로 이해하려면 다양한 Circuit을 테스트해야 합니다. QTS의 성능은 최적화되는 Circuit의 유형에 따라 크게 달라질 수 있습니다. 더 광범위한 비교를 위해 위의 벤치마크를 참고하거나 [블로그](https://www.ibm.com/quantum/blog/qiskit-performance)를 방문하세요.
## Step 3: Qiskit 프리미티브를 사용한 실행
이 튜토리얼은 트랜스파일에 초점을 맞추고 있으므로, 양자 디바이스에서 실험을 실행하지 않습니다. 목표는 Step 2의 최적화를 활용하여 깊이(depth) 또는 게이트 수가 줄어든 트랜스파일된 Circuit을 얻는 것입니다.
## Step 4: 결과를 원하는 고전 형식으로 후처리하여 반환
이 노트북에서는 실행이 없으므로 후처리할 결과도 없습니다.
# 파트 II. 트랜스파일된 Circuit 분석 및 벤치마킹
이 섹션에서는 트랜스파일된 Circuit을 분석하고 원본 버전과 더 상세하게 비교하는 방법을 보여줍니다. Circuit 깊이, 게이트 수, 트랜스파일 시간과 같은 지표에 집중하여 최적화의 효과를 평가합니다. 또한 다양한 Circuit 유형에 따라 결과가 어떻게 달라질 수 있는지 논의하고, 여러 시나리오에서 Transpiler의 전반적인 성능에 대한 통찰을 제공합니다.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts: {qubit_range_hw}"
)

Created 25 circuits with qubit counts: [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>